# TP53 and SRSF2 mutant cells

1)  Where do TP53 and SRSF2 mutant cells sit on the UMAP, does the position of these cells change with treatment?


      • Would you be able to make UMAP plots for the 5 patients where we have mutation data. I would like to see how the WT and MT cells change with time, therefore, I'd a UMAP plot for each of the three timepoints where they were measured. Can you have patients without the mutation in grey. Then put the other three outcomes (WT, MT and unknown) in unique colors.


      • There might be some interesting questions which arise from the analysis of WT and mutant cells.

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import warnings 
import numpy as np
import pandas as pd
import seaborn as sns


warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, facecolor='white')

In [ ]:
adata = sc.read_h5ad("HSC_gex_MDSpaper_figure2_newcelltypes_updatedtime.h5ad")

In [ ]:
adata

In [ ]:
adata.obs['timepoint_coarse'].value_counts()

In [ ]:
new_adata_HSC_nohealthy = adata[adata.obs["patient"].str.startswith("61")]
new_adata_HSC_nohealthy.obs["disease_state"].value_counts()

In [ ]:
#check if barcodes are duplicated
new_adata_HSC_nohealthy.obs_names.duplicated().sum()

In [ ]:
sc_mut = pd.read_csv("data/single_cell_mutation_for_priscilla.csv")

In [ ]:
sc_mut = sc_mut.iloc[:,1:]

In [ ]:
sc_mut.set_index("barcode_index", inplace=True)

In [ ]:
sc_mut.fillna("na", inplace=True)
sc_mut

In [ ]:
sc_mut = sc_mut.drop(columns=["BARCODE"])

In [ ]:
missing_ids = sc_mut.index.difference(new_adata_HSC_nohealthy.obs.index)
if not missing_ids.empty:
    print("Warning: The following IDs are in df but not in adata.obs:", missing_ids)

# Proceed with reindexing or merging
sc_mut = sc_mut.reindex(new_adata_HSC_nohealthy.obs.index).fillna("na")
adata_sc_mut = new_adata_HSC_nohealthy.copy()
adata_sc_mut.obs = adata_sc_mut.obs.join(sc_mut)
adata_sc_mut.obs

In [ ]:
adata_sc_mut.obs[['timepoint','timepoint_coarse']].value_counts()

In [ ]:
adata_sc_mut[adata_sc_mut.obs['chr17_76736877_G_A_call'].dropna().index,:].obs[['dataset_name','patient_alias']].value_counts()

In [ ]:
adata_sc_mut

In [ ]:
adata_sc_mut.obs

In [ ]:
print(adata_sc_mut.obs.dtypes)

In [ ]:
adata_sc_mut.obs[adata_sc_mut.obs.select_dtypes('object').columns] = adata_sc_mut.obs[adata_sc_mut.obs.select_dtypes('object').columns].astype(str).astype("category")

In [ ]:
adata_sc_mut.obs

In [ ]:
#adata_sc_mut.write_h5ad("09_adata_sc_mut.h5ad")

In [ ]:
newdf=pd.DataFrame({
    "X_coord_umap":adata_sc_mut.obsm["X_umap"][:,0],
    "Y_coord_umap":adata_sc_mut.obsm["X_umap"][:,1],
    "celltype":adata_sc_mut.obs["new_celltype"],
    "outcome_C6D28":adata_sc_mut.obs["outcome_C6D28"],
    'outcome_C12D29':adata_sc_mut.obs["outcome_C12D29"],
    "timepoint_coarse":adata_sc_mut.obs["timepoint_coarse"],
    "patient":adata_sc_mut.obs["patient"],
    "patient_alias": adata_sc_mut.obs["patient_alias"],
    'leiden': adata_sc_mut.obs["leiden"],
    'chr17_76736877_G_A_call': adata_sc_mut.obs['chr17_76736877_G_A_call'],
    'chr17_76736877_G_A_mut': adata_sc_mut.obs['chr17_76736877_G_A_mut'],
    'chr17_76736877_G_C_call': adata_sc_mut.obs['chr17_76736877_G_C_call'],
    'chr17_76736877_G_C_mut': adata_sc_mut.obs['chr17_76736877_G_C_mut'],
    'chr17_76736877_G_T_call': adata_sc_mut.obs['chr17_76736877_G_T_call'],
    'chr17_76736877_G_T_mut': adata_sc_mut.obs['chr17_76736877_G_T_mut'],
    'chr17_7674250_C_T_call': adata_sc_mut.obs['chr17_7674250_C_T_call'],
    'chr17_7674250_C_T_mut': adata_sc_mut.obs['chr17_7674250_C_T_mut'],
    'chr17_7675082_G_T_call': adata_sc_mut.obs['chr17_7675082_G_T_call'],
    'chr17_7675082_G_T_mut': adata_sc_mut.obs['chr17_7675082_G_T_mut'],
    'chr17_7676051_G_C_call': adata_sc_mut.obs['chr17_7676051_G_C_call'],
    'chr17_7676051_G_C_mut': adata_sc_mut.obs['chr17_7676051_G_C_mut']
})
newdf

In [ ]:
sc.pl.umap(adata_sc_mut, color=['chr17_76736877_G_A_call'])

In [ ]:
from MDS_figure2_dicts import *

In [ ]:
newdf['ctgrey'] = "#c8c8c8"

{'chr17:76736877_G>C': 'SRSF2',
 'chr17:7675082_G>T': 'TP53',
 'chr17:76736877_G>A': 'SRSF2',
 'chr17:76736877_G>T': 'SRSF2',
 'chr17:7674250_C>T': 'TP53',
 'chr17:7676051_G>C': 'TP53'}

In [ ]:
newdf[newdf['patient_alias'] == "P25"]

In [ ]:
newdf[(newdf['patient_alias'] == "P25") & (newdf['timepoint_coarse'] == 'mid')]['outcome_C6D28'].value_counts().idxmax()

In [ ]:
newdf['timepoint_coarse'].value_counts()

In [ ]:
newdf.columns[newdf.columns.str.contains('call')]

In [ ]:
# plot a umap for each patient and timepoint and mutation outcome
import matplotlib.lines as mlines
import re

size_mapping={'unknown':10,'wt':10,'mt':10}

for chr in list(newdf.columns[newdf.columns.str.contains('call')]):
    for p in list(set(newdf['patient_alias'])):

        fig,axes = plt.subplots(1,4, figsize=(28,7), dpi=300)

        for i, timepoint in enumerate(['pre','mid','post','progression']):
            ax = axes[i]
            
            #plot the dots for the desired categories
            df=newdf[newdf['patient_alias'] == p]
            data=df[df['timepoint_coarse'] == timepoint]
                
            #plot count in the top right hand corner of plot for easy cross checking
            counts = data[chr].value_counts()
            unknown_count = counts.get("unknown", 0)
            wt_count = counts.get("wt", 0)
            mt_count = counts.get("mt", 0)

            text_str = f"unknown: {unknown_count}\nwt: {wt_count}\nmt: {mt_count}"

            if wt_count == 0 and unknown_count == 0 and mt_count == 0 :
                fig.delaxes(ax)

            else:

                #plot the scatterplot for all cells
                ax.scatter(newdf['X_coord_umap'], newdf['Y_coord_umap'], c=newdf['ctgrey'], s=5)

                color_map = {'unknown':'grey','wt':'blue','mt':'red'}
                zorder_map = {'unknown':1,'wt':2,'mt':3}

                
                for category in color_map.keys():
                    subset = data[data[chr] == category]

                    sns.scatterplot(subset,
                    x='X_coord_umap',
                    y='Y_coord_umap',
                    ax=ax,
                    size=size_mapping[category],
                    color=color_map[category],
                    zorder=zorder_map[category],
                    edgecolor='none')

                ax.text(0.95,0.95,text_str, transform=ax.transAxes, ha="right", va="top", fontsize=12)

                ax.set_title(p+" "+timepoint, fontsize=25)
                ax.grid(False)
                ax.legend().set_visible(False)
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_xlabel('')
                ax.set_ylabel('')


        #create a dummy handle
        sh1 = mlines.Line2D([0], [0], marker='o', label='unknown',color='grey')
        sh2 = mlines.Line2D([0], [0], marker='o', label='wt',color='blue')
        sh3 = mlines.Line2D([0], [0], marker='o', label='mt',color='red')
        fig.suptitle(chr, fontsize=40)
        fig.legend(handles=[sh1, sh2, sh3], title='Mutations', markerscale=5,loc="upper center",bbox_to_anchor=(0.5,-0.1), fontsize=20,ncol=3) 
        fig.tight_layout()

        savefigname = re.sub(r"[^\w]", "_", str(chr))
        fig.savefig("figures/mutation_calls/"+savefigname+"_"+str(p)+"_UMAP_2.png", format='png', bbox_inches='tight')
        plt.close()
    


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import re
import matplotlib.font_manager as fm

#plot parameters

font_path = "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"  
arial_font = fm.FontProperties(fname=font_path)

plt.rcParams["font.family"] = arial_font.get_name()
plt.rcParams['pdf.fonttype'] = 'truetype'


size_mapping = {'unknown': 10, 'wt': 15, 'mt': 15}
color_map = {'unknown': 'green', 'wt': 'blue', 'mt': 'red'}
zorder_map = {'unknown': 1, 'wt': 2, 'mt': 3}

for chr in newdf.columns[newdf.columns.str.contains('call')]:
    for p in newdf['patient_alias'].unique():
        valid_axes_data = []  # Track valid subplots

        for timepoint in ['pre', 'mid', 'post', 'progression']:
            df = newdf[newdf['patient_alias'] == p]
            data = df[df['timepoint_coarse'] == timepoint]

            # Count mutations
            counts = data[chr].value_counts()
            unknown_count = counts.get("unknown", 0)
            wt_count = counts.get("wt", 0)
            mt_count = counts.get("mt", 0)

            # Skip subplot if all counts are zero
            if wt_count == 0 and unknown_count == 0 and mt_count == 0:
                continue

            valid_axes_data.append((timepoint, data))

        # If no valid subplots exist, skip this figure
        if not valid_axes_data:
            continue

        # Dynamically create figure with the correct number of subplots
        fig, axes = plt.subplots(1, len(valid_axes_data), figsize=(7 * len(valid_axes_data), 7), dpi=300)

        # Ensure axes is iterable (if only one subplot, `axes` is not a list)
        if len(valid_axes_data) == 1:
            axes = [axes]

        for ax, (timepoint, data) in zip(axes, valid_axes_data):
            # Plot background UMAP scatter
            ax.scatter(newdf['X_coord_umap'], newdf['Y_coord_umap'], c=newdf['ctgrey'], s=5)

            # Overlay mutation points
            for category in color_map.keys():
                subset = data[data[chr] == category]
                sns.scatterplot(subset, x='X_coord_umap', y='Y_coord_umap', ax=ax,
                                size=size_mapping[category], color=color_map[category],
                                zorder=zorder_map[category], edgecolor='none')

            # Display counts
            counts = data[chr].value_counts()
            text_str = f"unknown: {counts.get('unknown', 0)}\nwt: {counts.get('wt', 0)}\nmt: {counts.get('mt', 0)}"
            ax.text(0.95, 0.95, text_str, transform=ax.transAxes, ha="right", va="top", fontsize=15)

            ax.set_title(f"{p} {timepoint}", fontsize=25)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.grid(False)
            ax.legend().set_visible(False)

        # Adjust layout dynamically
        fig.suptitle(chr, fontsize=40)
        fig.tight_layout()

        # Create legend
        handles = [mlines.Line2D([0], [0], marker='o', label=label, color=color_map[label])
                   for label in color_map.keys()]
        fig.legend(handles=handles, title='Mutations', markerscale=5, loc="upper center",
                   bbox_to_anchor=(0.5, -0.1), fontsize=20, ncol=3)

        # Save figure as PDF
        savefigname = re.sub(r"[^\w]", "_", str(chr))
        fig.savefig(f"figures/mutation_calls/{savefigname}_{p}_UMAP.pdf", format='pdf', bbox_inches='tight')
        plt.close(fig)


In [ ]:
adata_sc_mut.obs[['patient_alias','specific_outcome_C12D29']].value_counts()